# 01 - Preprocessing and Dataset Demo

This notebook is for a demo video. It explains the frozen preprocessing design and displays verified dataset artifacts. It does **not** rerun preprocessing by default.

In [ ]:
from pathlib import Path
import sys

# Run notebooks from either the repository root or the notebooks/ folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Paper package:', PROJECT_ROOT / 'paper')

## What the preprocessing pipeline does

- Loads one EDF at a time.
- Verifies 18 common EEG channels.
- Applies 0.5--40 Hz bandpass filtering and 50 Hz notch filtering.
- Generates 4-second windows with 2-second stride.
- Applies per-window, per-channel z-score normalization.
- Writes per-EDF shard files under `data/interim/windows/`.

For the demo, we inspect the final paper artifacts instead of regenerating data.

In [ ]:
import pandas as pd
from IPython.display import display

dataset = pd.read_csv(PROJECT_ROOT / 'paper/tables/dataset_summary.csv')
splits = pd.read_csv(PROJECT_ROOT / 'paper/tables/patient_split_table.csv')
classes = pd.read_csv(PROJECT_ROOT / 'paper/tables/class_distribution_table.csv')

display(dataset)
display(splits)
display(classes.head(10))

## Pipeline and preprocessing figures

In [ ]:
from IPython.display import Image, display

for fig in ['pipeline_diagram.png', 'preprocessing_flowchart.png', 'data_flow_diagram.png', 'patient_split_diagram.png']:
    print(fig)
    display(Image(filename=str(PROJECT_ROOT / 'paper/figures' / fig)))

## Shard directory check

This checks whether local shard files exist. It does not load all EEG windows into RAM.

In [ ]:
from pathlib import Path

shard_dir = PROJECT_ROOT / 'data/interim/windows'
print('Shard directory exists:', shard_dir.exists())
if shard_dir.exists():
    x_files = sorted(shard_dir.glob('*_X.npy'))
    y_files = sorted(shard_dir.glob('*_y.npy'))
    meta_files = sorted(shard_dir.glob('*_meta.csv'))
    print('X shards:', len(x_files))
    print('y shards:', len(y_files))
    print('metadata shards:', len(meta_files))
    print('Example shards:', [p.name for p in x_files[:5]])
else:
    print('Shard directory is not present on this machine. Use the paper tables for demo results.')

## Optional heavy command - do not run during demo

The command below would regenerate preprocessing shards. It is intentionally not executed here.

In [ ]:
# Heavy / frozen pipeline command. Do not run during a results demo.
# !python src/data/preprocess_eeg.py